# WGAN Performance Benchmark: Original vs Optimized

This notebook compares the training time and results between:
- **Original package**: `/Users/anzony.quisperojas/Documents/GitHub/python/ds-wgan`
- **Optimized package**: `/Users/anzony.quisperojas/Documents/GitHub/python/dswganoptx`

In [ ]:
import pandas as pd
import numpy as np
import torch
from time import time
from copy import copy
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# Device to use
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## Load Data

In [ ]:
# Load the CPS dataset
df = pd.read_feather('/Users/anzony.quisperojas/Documents/GitHub/python/ds-wgan/data/original_data/cps.feather')
df = df.drop(["u74", "u75"], axis=1)

# Create balanced dataset (same as tutorial)
np.random.seed(42)  # For reproducibility
df_balanced = df.sample(2*len(df), weights=(1-df.t.mean())*df.t+df.t.mean()*(1-df.t), replace=True)

print(f"Original dataset shape: {df.shape}")
print(f"Balanced dataset shape: {df_balanced.shape}")

## Define Common Parameters

In [ ]:
# Model 0: X | t
continuous_vars_0 = ["age", "education", "re74", "re75"]
continuous_lower_bounds_0 = {"re74": 0, "re75": 0}
categorical_vars_0 = ["black", "hispanic", "married", "nodegree"]
context_vars_0 = ["t"]

# Model 1: Y | X, t
continuous_vars_1 = ["re78"]
continuous_lower_bounds_1 = {"re78": 0}
categorical_vars_1 = []
context_vars_1 = ["t", "age", "education", "re74", "re75", "black", "hispanic", "married", "nodegree"]

# Training parameters (same as tutorial)
BATCH_SIZE = 4096
MAX_EPOCHS = 1000
CRITIC_LR = 1e-3
GENERATOR_LR = 1e-3
PRINT_EVERY = 100

---
# Part 1: Original Package Benchmark
---

In [ ]:
# Import original package
import sys

# Remove optimized package from path if present
sys.path = [p for p in sys.path if 'dswganoptx' not in p]

# Add original package to path
sys.path.insert(0, '/Users/anzony.quisperojas/Documents/GitHub/python/ds-wgan')

# Force reimport
if 'wgan' in sys.modules:
    del sys.modules['wgan']
if 'wgan.wgan' in sys.modules:
    del sys.modules['wgan.wgan']

import wgan as wgan_original
print("Loaded ORIGINAL wgan package")
print(f"Package location: {wgan_original.__file__}")

In [ ]:
# Initialize original models
print("Initializing ORIGINAL models...")
print("=" * 60)

torch.manual_seed(42)  # For reproducibility

data_wrappers_orig = [
    wgan_original.DataWrapper(df_balanced, continuous_vars_0, categorical_vars_0,
                              context_vars_0, continuous_lower_bounds_0),
    wgan_original.DataWrapper(df_balanced, continuous_vars_1, categorical_vars_1,
                              context_vars_1, continuous_lower_bounds_1)
]

specs_orig = [
    wgan_original.Specifications(dw, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, 
                                 critic_lr=CRITIC_LR, generator_lr=GENERATOR_LR, 
                                 print_every=PRINT_EVERY, device=DEVICE)
    for dw in data_wrappers_orig
]

generators_orig = [wgan_original.Generator(spec) for spec in specs_orig]
critics_orig = [wgan_original.Critic(spec) for spec in specs_orig]

In [ ]:
# Train Model 0 with ORIGINAL package
print("\n" + "=" * 60)
print("ORIGINAL PACKAGE - Training Model 0: X | t")
print("=" * 60)

torch.manual_seed(42)
start_time_orig_0 = time()
x, context = data_wrappers_orig[0].preprocess(df_balanced)
wgan_original.train(generators_orig[0], critics_orig[0], x, context, specs_orig[0])
time_orig_model0 = time() - start_time_orig_0

print(f"\n>>> ORIGINAL Model 0 training time: {time_orig_model0:.2f} seconds")

In [ ]:
# Train Model 1 with ORIGINAL package
print("\n" + "=" * 60)
print("ORIGINAL PACKAGE - Training Model 1: Y | X, t")
print("=" * 60)

torch.manual_seed(42)
start_time_orig_1 = time()
x, context = data_wrappers_orig[1].preprocess(df_balanced)
wgan_original.train(generators_orig[1], critics_orig[1], x, context, specs_orig[1])
time_orig_model1 = time() - start_time_orig_1

print(f"\n>>> ORIGINAL Model 1 training time: {time_orig_model1:.2f} seconds")

In [ ]:
# Generate data with ORIGINAL package
print("\nGenerating data with ORIGINAL package...")

torch.manual_seed(42)
start_gen_orig = time()

df_generated_orig = data_wrappers_orig[0].apply_generator(generators_orig[0], df.sample(int(1e6), replace=True, random_state=42))
df_generated_orig = data_wrappers_orig[1].apply_generator(generators_orig[1], df_generated_orig)

# Counterfactuals
df_generated_cf_orig = copy(df_generated_orig)
df_generated_cf_orig["t"] = 1 - df_generated_cf_orig["t"]
df_generated_orig["re78_cf"] = data_wrappers_orig[1].apply_generator(generators_orig[1], df_generated_cf_orig)["re78"]

time_gen_orig = time() - start_gen_orig

# Compute ATT
att_orig = ((df_generated_orig.re78 - df_generated_orig.re78_cf) * (2*df_generated_orig.t - 1))[df_generated_orig.t == 1].mean()

print(f"ORIGINAL generation time (1M samples): {time_gen_orig:.2f} seconds")
print(f"ORIGINAL ATT: {att_orig:.2f}")

---
# Part 2: Optimized Package Benchmark
---

In [ ]:
# Import optimized package
import sys

# Remove original package from path
sys.path = [p for p in sys.path if 'ds-wgan' not in p or 'dswganoptx' in p]

# Add optimized package to path
sys.path.insert(0, '/Users/anzony.quisperojas/Documents/GitHub/python/dswganoptx')

# Force reimport
if 'wgan' in sys.modules:
    del sys.modules['wgan']
if 'wgan.wgan' in sys.modules:
    del sys.modules['wgan.wgan']

import wgan as wgan_optimized
print("Loaded OPTIMIZED wgan package")
print(f"Package location: {wgan_optimized.__file__}")

In [ ]:
# Initialize optimized models
print("Initializing OPTIMIZED models...")
print("=" * 60)

torch.manual_seed(42)  # For reproducibility

data_wrappers_opt = [
    wgan_optimized.DataWrapper(df_balanced, continuous_vars_0, categorical_vars_0,
                               context_vars_0, continuous_lower_bounds_0),
    wgan_optimized.DataWrapper(df_balanced, continuous_vars_1, categorical_vars_1,
                               context_vars_1, continuous_lower_bounds_1)
]

specs_opt = [
    wgan_optimized.Specifications(dw, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, 
                                  critic_lr=CRITIC_LR, generator_lr=GENERATOR_LR, 
                                  print_every=PRINT_EVERY, device=DEVICE)
    for dw in data_wrappers_opt
]

generators_opt = [wgan_optimized.Generator(spec) for spec in specs_opt]
critics_opt = [wgan_optimized.Critic(spec) for spec in specs_opt]

In [ ]:
# Train Model 0 with OPTIMIZED package
print("\n" + "=" * 60)
print("OPTIMIZED PACKAGE - Training Model 0: X | t")
print("=" * 60)

torch.manual_seed(42)
start_time_opt_0 = time()
x, context = data_wrappers_opt[0].preprocess(df_balanced)
wgan_optimized.train(generators_opt[0], critics_opt[0], x, context, specs_opt[0])
time_opt_model0 = time() - start_time_opt_0

print(f"\n>>> OPTIMIZED Model 0 training time: {time_opt_model0:.2f} seconds")

In [ ]:
# Train Model 1 with OPTIMIZED package
print("\n" + "=" * 60)
print("OPTIMIZED PACKAGE - Training Model 1: Y | X, t")
print("=" * 60)

torch.manual_seed(42)
start_time_opt_1 = time()
x, context = data_wrappers_opt[1].preprocess(df_balanced)
wgan_optimized.train(generators_opt[1], critics_opt[1], x, context, specs_opt[1])
time_opt_model1 = time() - start_time_opt_1

print(f"\n>>> OPTIMIZED Model 1 training time: {time_opt_model1:.2f} seconds")

In [ ]:
# Generate data with OPTIMIZED package
print("\nGenerating data with OPTIMIZED package...")

torch.manual_seed(42)
start_gen_opt = time()

df_generated_opt = data_wrappers_opt[0].apply_generator(generators_opt[0], df.sample(int(1e6), replace=True, random_state=42))
df_generated_opt = data_wrappers_opt[1].apply_generator(generators_opt[1], df_generated_opt)

# Counterfactuals
df_generated_cf_opt = copy(df_generated_opt)
df_generated_cf_opt["t"] = 1 - df_generated_cf_opt["t"]
df_generated_opt["re78_cf"] = data_wrappers_opt[1].apply_generator(generators_opt[1], df_generated_cf_opt)["re78"]

time_gen_opt = time() - start_gen_opt

# Compute ATT
att_opt = ((df_generated_opt.re78 - df_generated_opt.re78_cf) * (2*df_generated_opt.t - 1))[df_generated_opt.t == 1].mean()

print(f"OPTIMIZED generation time (1M samples): {time_gen_opt:.2f} seconds")
print(f"OPTIMIZED ATT: {att_opt:.2f}")

---
# Part 3: Results Comparison
---

In [ ]:
# Calculate speedups
speedup_model0 = time_orig_model0 / time_opt_model0
speedup_model1 = time_orig_model1 / time_opt_model1
speedup_total_train = (time_orig_model0 + time_orig_model1) / (time_opt_model0 + time_opt_model1)
speedup_gen = time_gen_orig / time_gen_opt

print("\n" + "=" * 70)
print("                    PERFORMANCE COMPARISON RESULTS")
print("=" * 70)
print(f"\nDevice: {DEVICE}")
print(f"Epochs: {MAX_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print("\n" + "-" * 70)
print("                         TRAINING TIME (seconds)")
print("-" * 70)
print(f"{'Component':<25} {'Original':>12} {'Optimized':>12} {'Speedup':>12}")
print("-" * 70)
print(f"{'Model 0 (X|t)':<25} {time_orig_model0:>12.2f} {time_opt_model0:>12.2f} {speedup_model0:>11.2f}x")
print(f"{'Model 1 (Y|X,t)':<25} {time_orig_model1:>12.2f} {time_opt_model1:>12.2f} {speedup_model1:>11.2f}x")
print("-" * 70)
print(f"{'TOTAL TRAINING':<25} {time_orig_model0+time_orig_model1:>12.2f} {time_opt_model0+time_opt_model1:>12.2f} {speedup_total_train:>11.2f}x")
print("\n" + "-" * 70)
print("                        GENERATION TIME (1M samples)")
print("-" * 70)
print(f"{'Data Generation':<25} {time_gen_orig:>12.2f} {time_gen_opt:>12.2f} {speedup_gen:>11.2f}x")
print("\n" + "-" * 70)
print("                           RESULTS COMPARISON")
print("-" * 70)
print(f"{'Metric':<25} {'Original':>12} {'Optimized':>12} {'Difference':>12}")
print("-" * 70)
print(f"{'ATT Estimate':<25} {att_orig:>12.2f} {att_opt:>12.2f} {abs(att_orig-att_opt):>12.2f}")
print("=" * 70)

In [ ]:
# Time saved
time_saved_train = (time_orig_model0 + time_orig_model1) - (time_opt_model0 + time_opt_model1)
time_saved_gen = time_gen_orig - time_gen_opt
time_saved_total = time_saved_train + time_saved_gen

pct_saved_train = (time_saved_train / (time_orig_model0 + time_orig_model1)) * 100
pct_saved_gen = (time_saved_gen / time_gen_orig) * 100 if time_gen_orig > 0 else 0

print("\n" + "=" * 70)
print("                         TIME SAVINGS SUMMARY")
print("=" * 70)
print(f"\nTraining time saved: {time_saved_train:.2f} seconds ({pct_saved_train:.1f}% reduction)")
print(f"Generation time saved: {time_saved_gen:.2f} seconds ({pct_saved_gen:.1f}% reduction)")
print(f"\nTOTAL TIME SAVED: {time_saved_total:.2f} seconds")
print("=" * 70)

In [ ]:
# Visual comparison
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Training time comparison
categories = ['Model 0\n(X|t)', 'Model 1\n(Y|X,t)', 'Total\nTraining']
orig_times = [time_orig_model0, time_orig_model1, time_orig_model0 + time_orig_model1]
opt_times = [time_opt_model0, time_opt_model1, time_opt_model0 + time_opt_model1]

x = np.arange(len(categories))
width = 0.35

bars1 = axes[0].bar(x - width/2, orig_times, width, label='Original', color='#ff7f0e')
bars2 = axes[0].bar(x + width/2, opt_times, width, label='Optimized', color='#2ca02c')
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Training Time Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Generation time comparison
axes[1].bar(['Original', 'Optimized'], [time_gen_orig, time_gen_opt], 
            color=['#ff7f0e', '#2ca02c'])
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('Generation Time (1M samples)')
axes[1].grid(axis='y', alpha=0.3)

# Speedup visualization
speedups = [speedup_model0, speedup_model1, speedup_total_train, speedup_gen]
speedup_labels = ['Model 0', 'Model 1', 'Total Train', 'Generation']
colors = ['#1f77b4' if s >= 1 else '#d62728' for s in speedups]

bars = axes[2].bar(speedup_labels, speedups, color=colors)
axes[2].axhline(y=1, color='black', linestyle='--', linewidth=1, label='No change')
axes[2].set_ylabel('Speedup (x times faster)')
axes[2].set_title('Speedup Factor')
axes[2].grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, speedups):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                 f'{val:.2f}x', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('/Users/anzony.quisperojas/Documents/GitHub/python/dswganoptx/benchmark_results.png', dpi=150)
plt.show()

print("\nBenchmark plot saved to: benchmark_results.png")

## Distribution Comparison

Verify that both packages produce similar distributions.

In [ ]:
# Compare statistics of generated data
print("\n" + "=" * 70)
print("            GENERATED DATA STATISTICS COMPARISON")
print("=" * 70)

compare_cols = ['age', 'education', 're74', 're75', 're78']

print(f"\n{'Variable':<12} {'Metric':<8} {'Original':>12} {'Optimized':>12} {'Diff':>10}")
print("-" * 60)

for col in compare_cols:
    orig_mean = df_generated_orig[col].mean()
    opt_mean = df_generated_opt[col].mean()
    orig_std = df_generated_orig[col].std()
    opt_std = df_generated_opt[col].std()
    
    print(f"{col:<12} {'Mean':<8} {orig_mean:>12.2f} {opt_mean:>12.2f} {abs(orig_mean-opt_mean):>10.2f}")
    print(f"{'':<12} {'Std':<8} {orig_std:>12.2f} {opt_std:>12.2f} {abs(orig_std-opt_std):>10.2f}")

print("\nNote: Small differences are expected due to random sampling in training.")

In [ ]:
# Final summary
print("\n" + "#" * 70)
print("#" + " " * 68 + "#")
print("#" + "               BENCHMARK COMPLETE - FINAL SUMMARY".center(68) + "#")
print("#" + " " * 68 + "#")
print("#" * 70)
print(f"\n  Overall Training Speedup: {speedup_total_train:.2f}x faster")
print(f"  Overall Generation Speedup: {speedup_gen:.2f}x faster")
print(f"  Total Time Saved: {time_saved_total:.1f} seconds")
print(f"\n  Results Match: ATT difference = {abs(att_orig - att_opt):.2f}")
print("\n" + "#" * 70)